# 02 — Driver behavior

**Audience:** drivers and the team. Answers: who drives smoothly, who slams the brakes, who improves over time?

All metrics computed per session from the cleaned `control_inputs` collection.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from etl.config import load_config
from etl.extract import iter_documents
from etl.transform import to_dataframe, add_derived_features, session_summary
from etl.load import get_collection

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
config = load_config()

In [ ]:
collection = get_collection(config.mongo_uri, config.mongo_db, config.mongo_collection)
df = add_derived_features(to_dataframe(list(iter_documents(collection))))
summary = session_summary(df)
summary.head(10)

## 1. Session leaderboard — smoothness vs aggression
Smooth driving = low steering-delta standard deviation. Aggressive = high mean absolute steering delta.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(summary['steering_smoothness'], summary['steering_aggression'], s=summary['rows']/50, alpha=0.7, c=summary['avg_throttle'], cmap='viridis')
ax.set_xlabel('Steering jitter (std dev of delta) — lower is smoother')
ax.set_ylabel('Steering aggression (mean |delta|)')
ax.set_title('Smoothness vs aggression per session (bubble = row count, color = avg throttle)')
plt.tight_layout()

## 2. Throttle/brake overlap — beginner indicator
How often is the driver pressing both throttle and brake at the same time? Pros don't.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
summary['throttle_brake_overlap_pct'].sort_values(ascending=False).head(20).plot.bar(ax=ax, color='#f97316')
ax.set_title('Top 20 sessions by throttle/brake overlap %')
ax.set_ylabel('% of samples with both > 5%')
plt.xticks(rotation=70, ha='right')
plt.tight_layout()

## 3. Time-series of one session
Pick a session ID from the summary table to drill in.

In [ ]:
# Most recent non-trivial session
candidates = summary[summary['rows'] > 100].head(1)
target_row = candidates.iloc[0] if not candidates.empty else summary.iloc[0]
TARGET_LABEL = target_row.name  # e.g. "S07"
TARGET_UUID = target_row['uuid']
plot_df = df[df['sessionId'] == TARGET_UUID]
print(f'Plotting {TARGET_LABEL} ({TARGET_UUID}) — {len(plot_df)} rows')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(plot_df.index, plot_df['steering'], color='#ef4444', linewidth=0.8)
axes[0].axhline(0, color='gray', linewidth=0.4, linestyle='--')
axes[0].set_ylabel('Steering')
axes[0].set_ylim(-1.1, 1.1)
axes[1].plot(plot_df.index, plot_df['throttle'], color='#22c55e', linewidth=0.8, label='Throttle')
axes[1].plot(plot_df.index, plot_df['brake'], color='#f97316', linewidth=0.8, label='Brake')
axes[1].set_ylabel('Throttle / Brake')
axes[1].legend(loc='upper right', fontsize=8)
axes[2].plot(plot_df.index, plot_df['movement'], color='#60a5fa', linewidth=0.8)
axes[2].axhline(0, color='gray', linewidth=0.4, linestyle='--')
axes[2].set_ylabel('Movement (throttle - brake)')
axes[2].set_ylim(-1.1, 1.1)
fig.suptitle(f'{TARGET_LABEL}', fontsize=12)
plt.tight_layout()

## 4. Steering symmetry — does the driver favor one side?

In [ ]:
active = df[df['steering'].abs() > 0.05]
left = (active['steering'] < 0).sum()
right = (active['steering'] > 0).sum()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['left', 'right'], [left, right], color=['#3b82f6', '#ef4444'])
ax.set_title(f'Steering symmetry — left: {left:,}  right: {right:,}  ratio: {left/max(right,1):.2f}')
plt.tight_layout()

## 5. Reaction time per session
Time from session start to first significant steering input.

In [ ]:
def first_input_delay(group):
    significant = group[group['abs_steering'] > 0.1]
    if significant.empty:
        return None
    return (significant.index.min() - group.index.min()).total_seconds()

reaction = df[df['sessionId'].notna()].groupby('sessionId').apply(first_input_delay, include_groups=False).dropna()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(reaction, bins=30, color='#8b5cf6', edgecolor='black', linewidth=0.3)
ax.set_title('Reaction time: session start → first significant steering input')
ax.set_xlabel('seconds')
plt.tight_layout()
print(f'Median reaction: {reaction.median():.2f}s  mean: {reaction.mean():.2f}s')